# LeukemiPrediag v2 — Train Fusion Model

This notebook evaluates the frozen ablation ladder:

- **M0**: static clinical/genetic baseline
- **M1**: M0 + longitudinal clone dynamics
- **M2**: M1 + NVIDIA BioNeMo / ESM-2 molecular representation

The purpose is to measure incremental value, not to claim clinical validation. Use an outcome-linked pre-diagnostic cohort for the leukemia-progression endpoint.


In [1]:
# Colab / Drive bootstrap — edit ROOT if your repo is elsewhere.
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/leukemia-prediagnose/blood-cancer-early-detection')
SRC = ROOT / 'src'
import sys
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))
print(ROOT)


/content/drive/MyDrive/leukemia-prediagnose/blood-cancer-early-detection


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

from src.clone_dynamics import patient_dynamics_features
from src.embedding import patient_embedding_matrix
from src.fusion_model import (
    build_feature_bundle, evaluate_ladder, bootstrap_increment,
    random_embedding_null, sensitivity_threshold, fit_final, save_artifact,
)
from src.longitudinal import make_landmark_labels


ModuleNotFoundError: No module named 'src'

## 1. Configure prepared inputs

Expected patient table: one row per prediction index. For a real progression experiment it should contain either `label`, or `event` + `years_to_event_or_censor` so the notebook can create a fixed-horizon landmark label.

Serial table should contain: `patient_id`, `variant_id`, `time_years`, `vaf`.

Mutation table should contain: `patient_id`, `mut_key`, `vaf`.

Embedding table should contain: `mut_key`, `delta`, where `delta` is the BioNeMo/ESM-2 residue-delta vector.


In [3]:
# EDIT THESE PATHS to match the cohort you are evaluating.
PATIENTS_CSV = ROOT / 'data/processed/patients_outcomes.csv'
SERIAL_CSV = ROOT / 'data/processed/serial_vaf.csv'
MUTATIONS_CSV = ROOT / 'data/processed/mutations.csv'
EMBEDDINGS_PKL = ROOT / 'models/pooled_missense_with_delta.pkl'
OUTDIR = ROOT / 'results/fusion_v2'
OUTDIR.mkdir(parents=True, exist_ok=True)

HORIZON_YEARS = 5.0
TARGET_SPECIFICITY = 0.98
N_SPLITS = 5
N_PCA = 16
GENES = []  # optionally fill with your frozen panel genes


In [ ]:
patients = pd.read_csv(PATIENTS_CSV)
patients['patient_id'] = patients['patient_id'].astype(str)

if 'label' not in patients.columns:
    patients = make_landmark_labels(
        patients,
        time_col='years_to_event_or_censor',
        event_col='event',
        horizon=HORIZON_YEARS,
    )

print('patients:', patients.shape, 'positive:', int(patients['label'].sum()))
patients.head()


In [ ]:
serial = pd.read_csv(SERIAL_CSV)
serial['patient_id'] = serial['patient_id'].astype(str)
dynamics = patient_dynamics_features(serial)
dynamics.to_csv(OUTDIR / 'patient_dynamics_features.csv', index=False)
print(dynamics.shape)
dynamics.head()


In [ ]:
mut = pd.read_csv(MUTATIONS_CSV)
mut['patient_id'] = mut['patient_id'].astype(str)
emb = pd.read_pickle(EMBEDDINGS_PKL)

# Existing helper returns [protein vector | has_embedding flag].
patient_emb = patient_embedding_matrix(
    mut, emb, patients['patient_id'].astype(str).tolist(), weight_by_vaf=True
)
protein = patient_emb[:, :-1]
patients = patients.copy()
patients['has_embedding'] = patient_emb[:, -1]
print('BioNeMo patient matrix:', protein.shape)


In [ ]:
bundle = build_feature_bundle(
    patients,
    dynamics=dynamics,
    protein_matrix=protein,
    genes=GENES,
)
y = patients['label'].to_numpy(int)

summary, preds, folds = evaluate_ladder(
    bundle, y,
    n_splits=N_SPLITS,
    n_pca=N_PCA,
    target_specificity=TARGET_SPECIFICITY,
)
summary


## 2. Incremental-value tests

The investor/scientific story is not the absolute M2 number alone. The important questions are:

- Does clone dynamics improve over static baseline? `M1 - M0`
- Does BioNeMo improve over dynamics + clinical context? `M2 - M1`
- Is the BioNeMo gain larger than a random-embedding null?


In [ ]:
inc_clone = bootstrap_increment(
    y, preds['M0_static'], preds['M1_plus_dynamics'], n_boot=2000
)
inc_bionemo = bootstrap_increment(
    y, preds['M1_plus_dynamics'], preds['M2_plus_BioNeMo'], n_boot=2000
)
null = random_embedding_null(
    bundle, y, n_reps=10, n_splits=N_SPLITS, n_pca=N_PCA,
    target_specificity=TARGET_SPECIFICITY,
)
print('clone increment:', inc_clone)
print('BioNeMo increment:', inc_bionemo)
print('random embedding null:', null)


In [ ]:
# Save OOF evidence for the deck / paper appendix.
oof = pd.DataFrame({
    'patient_id': patients['patient_id'],
    'label': y,
    **{f'p_{k}': v for k, v in preds.items()},
})
oof.to_csv(OUTDIR / 'oof_predictions.csv', index=False)
summary.to_csv(OUTDIR / 'fusion_ladder_summary.csv')
for name, ft in folds.items():
    ft.to_csv(OUTDIR / f'fold_metrics_{name}.csv')


In [ ]:
# Freeze the highest-level research artifact only after the evaluation design is fixed.
level = 'M2_plus_BioNeMo'
threshold = sensitivity_threshold(y, preds[level], TARGET_SPECIFICITY)
n_tab = bundle.X_static.shape[1] + bundle.X_dynamics.shape[1]
artifact = fit_final(
    bundle.X_m2, y,
    n_tabular=n_tab,
    n_protein=bundle.X_protein.shape[1],
    n_pca=N_PCA,
    threshold=threshold,
    feature_level=level,
    static_names=bundle.static_names,
    dynamics_names=bundle.dynamics_names,
    genes=GENES,
    target_specificity=TARGET_SPECIFICITY,
)
model_path = OUTDIR / 'leukemiprediag_fusion_v2.joblib'
meta_path = save_artifact(
    artifact, str(model_path),
    extra_meta={'horizon_years': HORIZON_YEARS,
                'research_use_only': True,
                'not_clinically_validated': True},
)
print(model_path)
print(meta_path)


## Interpretation gate

Do **not** move a number into the pitch deck as “leukemia prediction accuracy” unless the cohort was sampled before disease and the outcome/follow-up design is defensible. On LBC/SardiNIA without future leukemia outcomes, use this notebook to validate measurement/dynamics machinery, not clinical progression performance.
